In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, TensorDataset, DataLoader
from tab_transformer_pytorch import TabTransformer, FTTransformer
from preprocessing import get_features_and_target
from sklearn.preprocessing import LabelEncoder, StandardScaler
from RMSELoss import RMSELoss
import plotly.graph_objects as go
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion
from sklearn.model_selection import train_test_split

In [2]:
import huggingface_hub
huggingface_hub.login()

# Getting Dataframe

In [3]:
# Load the training and development datasets
train_df = pd.read_csv("data/train_data.csv")
dev_df = pd.read_csv("data/development_data.csv")
sc = StandardScaler()

target_column = "PullTest (N)"  

x_train, y_train = get_features_and_target(train_df, target_column)
x_dev, y_dev = get_features_and_target(dev_df, target_column)



# Fit Model

In [9]:
# columns that vary within a sample
time_series_cols = ["Force (N)", "Current (A)"]

# static columns
static_cols = ["Pressure (PSI)", "Welding Time (ms)", "Angle (Deg)", "Thickness A (mm)", "Thickness B (mm)"]

agg_df_train = train_df.groupby("Sample ID")[time_series_cols].agg(
    ['mean', 'std', 'min', 'max']
)
agg_df_dev = dev_df.groupby("Sample ID")[time_series_cols].agg(
    ['mean', 'std', 'min', 'max']
)

# flatten multi-index columns
agg_df_train.columns = ['_'.join(col) for col in agg_df_train.columns]
agg_df_dev.columns = ['_'.join(col) for col in agg_df_dev.columns]

# --- Add static columns (take first value per sample) --- 
static_df_train = train_df.groupby("Sample ID")[static_cols].first() 
static_df_dev = dev_df.groupby("Sample ID")[static_cols].first() 

 # Total Welding Time, adds the Welding Time Cycles in a single sample
weld_time_sum_train = train_df.groupby("Sample ID")["Welding Time (ms)"].sum() 
weld_time_sum_train = weld_time_sum_train.rename("Welding_Time_Total") 
weld_time_sum_dev = dev_df.groupby("Sample ID")["Welding Time (ms)"].sum() 
weld_time_sum_dev = weld_time_sum_dev.rename("Welding_Time_Total") 

# Combination of datasets
join_static_df_train = agg_df_train.join(static_df_train) 
x_train = join_static_df_train.join(weld_time_sum_train) 
join_static_df_dev = agg_df_dev.join(static_df_dev) 
x_dev = join_static_df_dev.join(weld_time_sum_dev) 

# Target value
y_train = train_df.groupby("Sample ID")['PullTest (N)'].first()

#Drop columns "Force (N)_std", "Current (A)_std" as ther is no variance with a single row so column is NaN
cols_to_drop = ["Force (N)_std", "Current (A)_std"] 

x_train = x_train.drop(columns=cols_to_drop) 
x_dev = x_dev.drop(columns=cols_to_drop)

try:
    x_train_scaled = sc.fit_transform(X=x_train)
    x_dev_scaled = sc.transform(x_dev)

    regressor = TabPFNRegressor()

    regressor.fit(x_train_scaled,y_train)

    # Predict on the test set
    predictions = regressor.predict(x_dev_scaled)

except Exception as e:
    print(f"No Scaling this time")

    regressor = TabPFNRegressor()

    regressor.fit(x_train,y_train)

    # Predict on the test set
    predictions = regressor.predict(x_dev)

# Check Validation Data

In [19]:
import plotly.graph_objects as go
import numpy as np

# Convert to numpy arrays
true_vals = np.array(y_dev).ravel()
pred_vals = np.array(predictions).ravel()

# Sample index
sample_idx = np.arange(len(true_vals))

# Category array (must be aligned with y_dev)
categories = dev_df.groupby("Sample ID")["Category"].first().values

# Masks for each category
mask_good    = categories == "Good"
mask_bad     = categories == "Bad"
mask_explode = categories == "Explode"

fig = go.Figure()

# --- GOOD (circles) ---
fig.add_trace(go.Scatter(
    x=sample_idx[mask_good],
    y=true_vals[mask_good],
    mode="markers",
    name="Good (True)",
    marker=dict(symbol="circle", color="red", size=7)
))

fig.add_trace(go.Scatter(
    x=sample_idx[mask_good],
    y=pred_vals[mask_good],
    mode="markers",
    name="Good (Pred)",
    marker=dict(symbol="circle", color="blue", size=7)
))

# --- BAD (X) ---
fig.add_trace(go.Scatter(
    x=sample_idx[mask_bad],
    y=true_vals[mask_bad],
    mode="markers",
    name="Bad (True)",
    marker=dict(symbol="x", color="red", size=9)
))

fig.add_trace(go.Scatter(
    x=sample_idx[mask_bad],
    y=pred_vals[mask_bad],
    mode="markers",
    name="Bad (Pred)",
    marker=dict(symbol="x", color="blue", size=9)
))

# --- EXPLODE (triangle-up) ---
fig.add_trace(go.Scatter(
    x=sample_idx[mask_explode],
    y=true_vals[mask_explode],
    mode="markers",
    name="Explode (True)",
    marker=dict(symbol="triangle-up", color="red", size=9)
))

fig.add_trace(go.Scatter(
    x=sample_idx[mask_explode],
    y=pred_vals[mask_explode],
    mode="markers",
    name="Explode (Pred)",
    marker=dict(symbol="triangle-up", color="blue", size=9)
))

# Optional: connecting lines for each sample
for i in range(len(sample_idx)):
    fig.add_trace(go.Scatter(
        x=[sample_idx[i], sample_idx[i]],
        y=[true_vals[i], pred_vals[i]],
        mode="lines",
        line=dict(color="gray", width=1),
        showlegend=False
    ))

fig.update_layout(
    title="Validation Samples: True vs Prediction (TabPFN) by Category",
    xaxis_title="Sample Index",
    yaxis_title="Pull Force",
    template="seaborn"
)

fig.show()


In [6]:
print(len(predictions))

99


# Check Validation Loss and R2

In [11]:
# Calculate MAE and RMSE and R2
mae  = mean_absolute_error(y_dev, predictions)
rmse = np.sqrt(mean_squared_error(y_dev, predictions))
R2   = r2_score(y_dev, predictions)


print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {R2:.2f}")


MAE:  126.24
RMSE: 218.98
R2: 0.62
